# Develop

## Import

In [1]:
import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score

from typing import List

## Model learning

In [2]:
def train_in_chunks(files_list: List[str],
                    feature_cols: List[str] = None,
                    batchsize: int = 20000,
                    target_col: str = 'target',
                    cat_features: List[int] = None,
                    model: CatBoostClassifier = None):
    """
    Функция для последовательного обучения модели по чанкам. Принимает следующие аргументы:
    files_list - список с файлами для обучения модели (полные пути до файлов)
    batchsize - размер чанка (кол-во строк)
    target_col - название целевой фичи
    model - модель
    """
    if feature_cols is None:
        raise ValueError("Надо указать feature_cols - колонки для обучения")
    
    first_batch_flag = True
    for file in files_list:
        current_file = pq.ParquetFile(file)
        for batch in current_file.iter_batches(batch_size=batchsize):
            chunk = batch.to_pandas()
            X_batch = chunk[feature_cols]
            y_batch = chunk[target_col]
            
            if first_batch_flag:
                model.fit(X_batch, y_batch, cat_features=cat_features)
                first_batch_flag = False
            else:
                model.fit(X_batch, y_batch, init_model=model)
    
    model.save_model('../models/catboost_model.cbm')

In [6]:
train_files = ['../datasets/joined/train_data.parquet'] # здесь указать полные пути размеченных train'ов
'''
cat_features = ['customer_id', 'event_type_nm','event_desc','channel_indicator_type',
                'channel_indicator_sub_type','currency_iso_cd',
                'mcc_code','pos_cd','accept_language','browser_language',
                'timezone','operating_system_type',
                'device_system_version','screen_size','developer_tools',
                'phone_voip_call_state','web_rdp_connection','compromised']
'''
train_cols = ['customer_id', 'event_type_nm','event_desc','channel_indicator_type',
                'channel_indicator_sub_type','currency_iso_cd',
                'mcc_code','pos_cd','accept_language','browser_language',
                'timezone','operating_system_type',
                'device_system_version','screen_size','developer_tools',
                'phone_voip_call_state','web_rdp_connection','compromised', 'battery']

catboost_model = CatBoostClassifier()

train_in_chunks(files_list=train_files,
                feature_cols=train_cols,
                model=catboost_model,
                cat_features=None # Надо указать категорильаные признаки
                )

CatBoostError: Bad value for num_feature[non_default_doc_idx=843,feature_idx=8]="ru-ru": Cannot convert 'ru-ru' to float

## Prediction

In [ ]:
def save_file_results(id_events: pd.Series,
                      predictions: np.ndarray,
                      save_path: str,
                      chunk_num: int):
    """
    Сохранение предсказаний вместе с идентификаторами в Parquet файл
    Принимает:
    id_events - объект pd.Series, который формируется в функции predict_in_chunks.
        Содержит данные об id операций. Нужен для коммита
    predictions - массив нампая с предсказаниями
    save_path - полный путь сохранения parquet файла
    chunk_num - флаговая переменная для указания номера чанка.
        Если чанк под номером 0 (первый), то надо создать новую таблицу. Иначе выполнить соединение с существующей
    """
    result_df = pd.DataFrame({
        'event_id': id_events,
        'predict': predictions
    })
    
    # Конвертируем в Arrow Table
    table = pa.Table.from_pandas(result_df)
    
    if chunk_num == 0:
        # Первый чанк или файл не существует - создаем новый
        pq.write_table(
            table,
            save_path,
            compression='snappy',
            flavor='spark'  # для совместимости со Spark если нужно
        )
        print(f"Создан файл: {save_path} ({len(result_df)} записей)")
    else:
        # Добавляем данные в существующий файл
        with pq.ParquetWriter(
            save_path,
            table.schema,
            compression='snappy',
            flavor='spark',
            append=True  # Важно: append режим
        ) as writer:
            writer.write_table(table)
        
        print(f"Добавлено {len(result_df)} записей в {save_path}")

In [ ]:
def predict_in_chunks(file: str,
                      batchsize: int = 20000,
                      cols: List[str] = None,
                      return_type: str = 'class',  # 'proba' или 'class'
                      model: CatBoostClassifier = None,
                      save_path: List[str] = '../datasets/validation_result.parquet'
                      ):
    """
    Функция для предсказания на больших файлах по чанкам. Принимает:
    file - файл предсказания (полный путь)
    batchsize - размер чанка
    cols - колонки, необходимые для обучения
    return_type - 'proba' для вероятностей, 'class' для классов
    model - используемая для предсказания модель
    save_path - полный путь для сохранения
    """
    if cols is None:
        raise AttributeError('Необходимо ввести параметр с колонками для предикта\n'
                             'Те же, что и использовались с трейне. Передай список колонок в параметр cols')
    if model is None:
        raise AttributeError('Необходимо ввести параметр с моделью для предикта')
    total_rows = 0

    print(f"Обработка файла: {file}")
    
    current_file = pq.ParquetFile(file)
    
    for batch_idx, batch in enumerate(current_file.iter_batches(batch_size=batchsize)):
        df_chunk = batch.to_pandas()
        chunk = df_chunk[cols]
        id_events=df_chunk['id_event']
        
        if return_type == 'proba':
            preds = model.predict_proba(chunk)[:, 1]
        else:
            preds = model.predict(chunk)
        
        save_file_results(id_events, preds, save_path, chunk_num=batch_idx)
        
        total_rows += len(chunk) 
        if batch_idx % 10 == 0:
            print(f"Обработано чанков: {batch_idx + 1}, строк: {total_rows}")

    print(f"Результаты сохранены в {save_path}")

In [ ]:
predict_in_chunks(file='labeled_train_part_3.parquet',
                  file_group='labeled_train',
                  return_type='class',
                  model=catboost_model,
                  save_path= '../datasets/validation_result.parquet')

## Error Analysis

In [ ]:
# Кастомная функция потерь, которая будет сильно штрафовать за ложно-отрицательные резы

In [ ]:
# Кастомная реализация sklearn.average_precision_score

In [ ]:
def target_class_mistakes_finder(predict_path: str,
                                 answers_path: str,
                                 predict_column: str,
                                 answers_column: str,
                                 id_column: str = 'event_id', 
                                 target_class: int = 0):
    """
    Функция по поиску расхождений предикта с известными данными
    именно с целевым классом - False Negative результаты
    Принимает:
    predict_path - полный путь к parquet файлу предсказания
    answers_path - полный путь к parquet файлу с ответами
    predcit_column - название колонки с лейблами предсказаний
    answers_column - название колонки с лейблами ответов
    id_column - название колонки для вывода
    target_class - фильтр по классу
    """
    # Создаем Spark сессию
    spark = SparkSession.builder \
        .appName("FindFalseNegatives") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()

    try:
        # Загружаем данные и проверяем колонки
        predictions = spark.read.parquet(predict_path)
        answers = spark.read.parquet(answers_path)
        
        predict_cols = predictions.columns
        answers_cols = answers.columns
        
        if id_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {id_column}")
        if id_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {id_column}")
        if predict_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {predict_column}")
        if answers_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {answers_column}")
        
        # Объединяем данные
        merged = (predictions
                  .select(id_column, predict_column)
                  .join(answers.select(id_column, answers_column),
                        on=id_column,
                        how="inner")
                  )
        
        # Определяем условие для поиска ошибок и находим ошибки
        if target_class in [0, 1, -1]:
            # реальный ответ = target class, предсказание иное
            error_condition = (col(answers_column) == target_class) & (col(predict_column) != target_class)
        else:
            raise ValueError("target_class должен быть 0, 1 или -1")
        
        errors = merged.filter(error_condition)
        
        # Считаем статистику
        total_count = merged.count()
        mismatch_count = merged.filter(col(predict_column) != col(answers_column)).count()
        error_count = errors.count()
        
        print(f"Всего записей: {total_count}")
        print(f"Всего несоответствий: {mismatch_count}")
        print(f"Найдено целевых ошибок: {error_count}")
        
        if error_count > 0:
            print(f"\nПервые 10 {id_column} с ошибками:")
            errors.select(id_column).show(10)

        return [row[id_column] for row in errors.select(id_column).collect()]
        
    finally:
        spark.stop()
    

In [ ]:
# Поиск лучших параметров и отбор полезных признаков (shap?)

## Test Predict (подготовка решения)

In [ ]:
train_files = ['labeled_train_part_1.parquet', 'labeled_train_part_2.parquet', 'labeled_train_part_3.parquet']
catboost_model = CatBoostClassifier()
train_in_chunks(files_list=train_files, file_group='labeled_train', model=catboost_model)


predict_in_chunks(file='test.parquet',
                  file_group='',
                  return_type='class',
                  model=catboost_model,
                  save_path= '../submits/submit.parquet')

## Submit Creating (Создание сабмита)

In [ ]:
save_submit_path = '../submits/submit.csv'

def create_submit(save_path: str = save_submit_path,
                  take_path: str = '../submits/submit.parquet'):
    result_df = pd.read_parquet(take_path)
    
    row_count = len(result_df)
    if row_count != 633683:
        raise ValueError(f"Представленный датафрейм имеет иное количество строк: {row_count}. Такой сабмит не пройдет. Необходимо 633683 строк")
    
    if list(result_df.columns) != ['event_id', 'predict']:
        raise ValueError(f'Неверные названия столбиков: {result_df.columns}')
    
    with open(save_path, 'w') as file:
        result_df.to_csv(file, sep=',', index=False)

In [ ]:
create_submit()